## Refactoring is Calling

Here is another beautifully flawed Python script.
While the previous example suffered from "God Class" and architectural coupling, this one is an **ETL (Extract, Transform, Load)** script that falls into entirely different Python-specific traps. It’s functional, but it is a maintenance nightmare.

Now it is your turn in We Do session. You have *45min - 1h* to work on the problem

### 🛠️ Your New Refactoring Mission


- [ ] Don't start implementing immediately.
- [ ] First list the problems you see. There should be 6-9 categories.
- [ ] Iteratively fix them


In [3]:
import json
import csv

import pandas as pd

global_employee_database = []
total_processed = 0
errors_encountered = 0

def addEmployeeRecord(record, target_list=[]): 
    global total_processed
    global errors_encountered
    try:
        target_list.append(record)
        total_processed = total_processed + 1
    except Exception:
        pass

def parseDataFiles(file_list):
    if file_list != None:
        if len(file_list) > 0:
            for fileIndex in range(len(file_list)):
                fileName = file_list[fileIndex]
                if fileName.endswith(".csv"):
                    try:
                        f = open(fileName, 'r') 
                        lines = f.readlines()
                        for line in lines[1:]: 
                            parts = line.strip().split(',')
                            if len(parts) == 4:
                                empDict = {'id': parts[0], 'name': parts[1], 'dept': parts[2], 'salary': float(parts[3])}
                                if empDict['salary'] > 0:
                                    addEmployeeRecord(empDict)
                    except Exception:
                        global errors_encountered
                        errors_encountered += 1
                elif fileName.endswith(".json"):
                    try:
                        f2 = open(fileName, 'r') 
                        data = json.loads(f2.read())
                        for item in data:
                            if 'id' in item and 'name' in item and 'dept' in item and 'salary' in item:
                                if float(item['salary']) > 0:
                                    addEmployeeRecord(item)
                    except Exception:
                        errors_encountered += 1

def calculate_Bonus_and_Print_report():
    global global_employee_database
    
    engineering_employees = []
    sales_employees = []
    
    for i in range(len(global_employee_database)):
        if global_employee_database[i]['dept'] == 'Engineering':
            engineering_employees.append(global_employee_database[i])
        elif global_employee_database[i]['dept'] == 'Sales':
            sales_employees.append(global_employee_database[i])
            
    out_file = open("department_bonuses.txt", "w")

    
    out_file.write("--- ENGINEERING ---\n")
    for emp in engineering_employees:
        bonus = 0
        if emp['salary'] < 50000:
            bonus = emp['salary'] * 0.10
        elif emp['salary'] >= 50000 and emp['salary'] <= 100000:
            bonus = emp['salary'] * 0.05
        else:
            bonus = emp['salary'] * 0.02
            
        out_file.write(emp['name'] + " gets a bonus of $" + str(bonus) + "\n")
        
    out_file.write("--- SALES ---\n")
    for emp in sales_employees:
        bonus = 0
        if emp['salary'] < 50000:
            bonus = emp['salary'] * 0.15 
        elif emp['salary'] >= 50000 and emp['salary'] <= 100000:
            bonus = emp['salary'] * 0.07
        else:
            bonus = emp['salary'] * 0.03
        out_file.write(emp['name'] + " gets a bonus of $" + str(bonus) + "\n")


if __name__ == '__main__':
    print("Starting batch job...")
    
    with open('data1.csv', 'w') as f:
        f.write("id,name,dept,salary\n1,Alice,Engineering,60000\n2,Bob,Sales,45000\n")
    with open('data2.json', 'w') as f:
        f.write('[{"id": 3, "name": "Charlie", "dept": "Engineering", "salary": 120000}]')

    files_to_process = ['data1.csv', 'data2.json', 'missing_file.csv']
    
    parseDataFiles(files_to_process)
    calculate_Bonus_and_Print_report()
    
    print("Job Done. Processed: " + str(total_processed) + ", Errors: " + str(errors_encountered))


Starting batch job...
Job Done. Processed: 3, Errors: 1
